# GraphRAG T1-T8 Fine-Tune: Llama 3 8B (QLoRA, GPU)

**Purpose.** Fine-tune the same Llama 3 8B used in the GraphRAG_OpenFDA pipeline so the model learns the *behaviours* the Jain Phase 1 audit tests (T1-T8): grounded answering from retrieved excerpts, ambiguity handling, comparison discipline, abstention, and citation integrity.

**Base scripts.** Built on `code/adv_graphrag.py` and `code/v2_adv_rag.py` of https://github.com/75Dineshchandra/GraphRAG_OpenFDA
- The pipeline loads the LLM through Ollama as `llama3:8b` (temperature 0.1, num_ctx 8192). The Hugging Face twin of that exact model, used here for training, is `meta-llama/Meta-Llama-3-8B-Instruct`.
- The answer contract is copied from `build_prompt()` in adv_graphrag.py: answer ONLY from the excerpts, short bullets, every bullet ends with its source tag(s), drugs kept separate, and the exact abstention line below.
- ABSTAIN_TEXT (copied verbatim): `INSUFFICIENT FDA DATA: The retrieved label text does not contain this information.`

**Training data (no leakage).** NO OpenFDA / DailyMed label text is used for training, because the GraphRAG and the frozen 60-question audit are built on it. Data comes only from `FreedomIntelligence/medical-o1-reasoning-SFT` (English), reshaped into the pipeline's grounded format. No synthetic or generated questions are used (decision, 2026-11-16): train on the dataset, then measure the frozen audit per T-category to see what it fixes.

**What fine-tuning can and cannot fix**
| Test | Covered here? |
|---|---|
| T1 drug-specific grounded QA | Yes, grounded format |
| T2 symptom + safety answering | Answering half, yes (retrieval half is a pipeline fix) |
| T3 brand query | Light (already 66.7%) |
| T4 ambiguity | Not targeted directly; the audit will show |
| T5 comparison discipline | Not targeted directly; the audit will show |
| T6 abstention | Not targeted directly; the audit will show |
| T7 long-section recovery | NO - embedding/chunking failure, fixed in the pipeline, not here |
| T8 citation integrity | Not targeted directly; the audit will show |

**Outputs** are saved to Google Drive (folder `GraphRAG_FineTune`): training JSONL, LoRA adapter, merged model, GGUF file and the Ollama Modelfile. Mount the Drive with the account you want the model saved in (for example your GWU account in Colab).

## Step 0 - GPU check and installs
Run in Colab with a GPU runtime (Runtime > Change runtime type > GPU), or on any machine with a CUDA GPU. An 8B model in 4-bit QLoRA needs roughly 10-12 GB of VRAM minimum; 16 GB or more is comfortable.

In [2]:
import subprocess, sys, torch

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("WARNING: no GPU visible. This notebook is written for GPU training; on CPU it will be extremely slow.")

pkgs = ["transformers>=4.44", "peft>=0.12", "bitsandbytes>=0.43", "datasets>=2.20", "accelerate>=0.33", "sentencepiece", "protobuf"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=False)
print("installs done")

torch: 2.14.1+cu130 | CUDA available: True
GPU: NVIDIA L4 | VRAM GB: 23.7
installs done


## Step 1 - Mount Drive and set paths / config
The fine-tuned model is saved in the same Drive you mount here. In Colab, sign in with the Google account whose Drive you want (for example GWU) before running this cell.

In [3]:
import os, json, random

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception as e:
    print("Drive mount skipped (not in Colab):", e)

DRIVE_BASE = "/content/drive/MyDrive/GraphRAG_FineTune" if IN_COLAB else os.path.abspath("./graphrag_finetune_outputs")
DATA_DIR   = os.path.join(DRIVE_BASE, "data")
ADAPTER_DIR = os.path.join(DRIVE_BASE, "adapter_labeltune_8b")
MERGED_DIR  = os.path.join(DRIVE_BASE, "merged_labeltune_8b")
GGUF_DIR    = os.path.join(DRIVE_BASE, "gguf")
for d in [DATA_DIR, ADAPTER_DIR, MERGED_DIR, GGUF_DIR]:
    os.makedirs(d, exist_ok=True)
print("Outputs will be saved under:", DRIVE_BASE)

# ---- Config: mirrors the GraphRAG scripts ----
OLLAMA_MODEL_IN_PIPELINE = "llama3:8b"                       # LLM_MODEL in adv_graphrag.py / v2_adv_rag.py
BASE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"           # HF twin of Ollama llama3:8b
ABSTAIN_TEXT = "INSUFFICIENT FDA DATA: The retrieved label text does not contain this information."  # verbatim from adv_graphrag.py

SYSTEM_PROMPT = ("You answer ONLY from the FDA drug label excerpts below.\nRULES:\n"
    "1. Use only the excerpts. Never use outside knowledge.\n"
    "2. Write short bullet points. Every bullet ends with its source tag(s), e.g. [S2] or [S1][S3].\n"
    "3. Keep drugs separate: start each bullet with the drug name.\n"
    f"4. If the excerpts do not answer the question, reply exactly: {ABSTAIN_TEXT}\n"
    "5. Give no medical advice beyond what the label says.")

N_REASONING = 3000          # all training examples come from medical-o1 (dataset-only design)
SEED = 42
random.seed(SEED)
print("config set | base model:", BASE_MODEL)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs will be saved under: /content/drive/MyDrive/GraphRAG_FineTune
config set | base model: meta-llama/Meta-Llama-3-8B-Instruct


## Step 2 - Load the base model on GPU (4-bit, exactly the pipeline's Llama 3 8B)
`meta-llama` is a gated repo: set an `HF_TOKEN` Colab secret (or run the login cell interactively) with a token that has accepted the Llama 3 licence.

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", None)
if not hf_token:
    from huggingface_hub import login
    login()  # interactive fallback
    hf_token = None

BASE_MODEL = "NousResearch/Meta-Llama-3-8B-Instruct"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                             device_map="auto", token=hf_token, torch_dtype=torch.bfloat16)
model.config.use_cache = False
print("model loaded on:", next(model.parameters()).device)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

model loaded on: cuda:0


## Step 3 - Pull the dataset
`FreedomIntelligence/medical-o1-reasoning-SFT`, English subset. Fields: `Question`, `Complex_CoT` (the reasoning), `Response` (the final answer). No FDA label text, so the frozen audit stays uncontaminated.

In [5]:
from datasets import load_dataset

raw = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en")
print(raw)
ex = raw["train"][0]
for k in ["Question", "Complex_CoT", "Response"]:
    print("\n---", k, "---\n", str(ex[k])[:300])

README.md:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

medical_o1_sft.json: reconstructing file:   0%|          |  0.00B / 58.2MB            

medical_o1_sft.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Question', 'Complex_CoT', 'Response'],
        num_rows: 19704
    })
})

--- Question ---
 Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?

--- Complex_CoT ---
 Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?

But wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after 

--- Response ---
 The specific cardiac abnormality most likely to be found in this scenario is a patent foramen ovale (PFO). This condition could allow a blood clot from the venous system, such as one from a deep vein thrombosis in the leg, to bypass the lungs 

## Step 4 - Reshape medical-o1 into the pipeline's grounded format (~3,000 examples)
Each medical-o1 answer passage becomes excerpt `[S1]`, and the target answer keeps the reasoning (shortened) plus a cited bullet in the exact answer contract of `build_prompt()`. This teaches grounded answering and citation habit (T1/T2 style) with the reasoning style you wanted, without any label text.

In [6]:
def shorten(text, words):
    w = str(text).split()
    return " ".join(w[:words]) + (" ..." if len(w) > words else "")

rows = raw["train"].shuffle(seed=SEED)
part_a = []
for r in rows:
    if len(part_a) >= N_REASONING:
        break
    q, cot, resp = r["Question"].strip(), r["Complex_CoT"].strip(), r["Response"].strip()
    if not q or not cot or not resp:
        continue
    user = f"EXCERPTS:\n[S1] (medical-o1 | response) {shorten(resp, 220)}\n\nQUESTION: {q}\n\nANSWER:\n"
    assistant = f"Reasoning: {shorten(cot, 110)}\n\n- {shorten(resp, 90)} [S1]"
    part_a.append({"system": SYSTEM_PROMPT, "user": user, "assistant": assistant, "tag": "T1T2"})
print("training examples:", len(part_a))
print(part_a[0]["assistant"][:400])

training examples: 3000
Reasoning: Alright, so a Gingival Margin Trimmer, or GMT for short, is some sort of dental tool used during cavity prep. I need to figure out what that second number in its formula really means, especially when working with the distal gingival margin. Let's start with the basics about these numbers. The first number in any dental instrument formula usually tells us the blade width, measured in ten


## Step 5 - Assemble, save to Drive, tokenise (prompt masked, train on the answer only)

In [7]:
from collections import Counter
from torch.utils.data import Dataset

train_data = list(part_a)
random.shuffle(train_data)
print("total examples:", len(train_data), "| by tag:", Counter(d["tag"] for d in train_data))

jsonl_path = os.path.join(DATA_DIR, "train_t1t8.jsonl")
with open(jsonl_path, "w") as f:
    for d in train_data:
        f.write(json.dumps(d) + "\n")
print("saved:", jsonl_path)

MAX_LEN = 1024
def to_features(d):
    msgs = [{"role": "system", "content": d["system"]},
            {"role": "user", "content": d["user"]}]
    prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full_text = prompt_text + d["assistant"] + tokenizer.eos_token
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"][:MAX_LEN]
    labels = ([-100] * len(prompt_ids) + full_ids[len(prompt_ids):])[:MAX_LEN]
    return {"input_ids": full_ids, "labels": labels, "attention_mask": [1] * len(full_ids)}

features = [to_features(d) for d in train_data]
print("tokenised:", len(features), "| example length:", len(features[0]["input_ids"]))

class ListDataset(Dataset):
    def __len__(self): return len(features)
    def __getitem__(self, i): return features[i]

def collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    input_ids, labels, attn = [], [], []
    for b in batch:
        n = maxlen - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [pad] * n)
        labels.append(b["labels"] + [-100] * n)
        attn.append(b["attention_mask"] + [0] * n)
    return {"input_ids": torch.tensor(input_ids), "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(attn)}

train_ds = ListDataset()

total examples: 3000 | by tag: Counter({'T1T2': 3000})
saved: /content/drive/MyDrive/GraphRAG_FineTune/data/train_t1t8.jsonl
tokenised: 3000 | example length: 566


## Step 6 - QLoRA training on GPU
QLoRA exactly as scoped in the LabelTune plan: 4-bit base, LoRA adapters only, no full fine-tune. Batch 2 x grad-accum 8 = effective batch 16. Two epochs over ~3,000 examples is roughly 375 steps.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments

model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=12,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    gradient_checkpointing=True,
    report_to=[],
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate)
trainer.train()
print("training done")

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


Step,Training Loss


Step,Training Loss
10,1.123221
20,0.836904
30,0.785814
40,0.780209
50,0.780268
60,0.787838
70,0.765370
80,0.744626
90,0.756760
100,0.762212


## Step 7 - Save the adapter and a training summary to Drive

In [9]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
summary = {"base_model": BASE_MODEL, "pipeline_model_name": OLLAMA_MODEL_IN_PIPELINE,
           "dataset": "FreedomIntelligence/medical-o1-reasoning-SFT (en)",
           "n_examples": len(train_data), "by_tag": dict(Counter(d["tag"] for d in train_data)),
           "lora": {"r": 16, "alpha": 32, "dropout": 0.05}, "epochs": 2, "lr": 2e-4,
           "note": "No OpenFDA/DailyMed text in training (leakage rule). T7 not addressed by fine-tuning."}
with open(os.path.join(DRIVE_BASE, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("adapter + summary saved to:", DRIVE_BASE)

adapter + summary saved to: /content/drive/MyDrive/GraphRAG_FineTune


## Step 8 - Smoke test the tuned adapter on held-out dataset questions
These probes are medical-o1 questions from the tail of the shuffled dataset, well beyond the training slice, shown in the grounded format. Read them before packaging: the answers should reason over the excerpt and end with a cited bullet.

In [ ]:
model.config.use_cache = True
model.eval()

def probe(user_text, max_new=300):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_text}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new, temperature=0.1, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))
    print("=" * 80)

# Probes are held-out medical-o1 questions (tail of the shuffled split, beyond the training slice).
for r in [rows[-3], rows[-2], rows[-1]]:
    probe(f"EXCERPTS:\n[S1] (medical-o1 | response) {shorten(r['Response'], 220)}\n\nQUESTION: {r['Question']}\n\nANSWER:\n")

[transformers] Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reasoning: Alright, let's think about this. We've got a 47-year-old guy who's been drinking a lot and now he's got this really bad pain on the right side of his chest. He's also got a fever, chills, and he's sweating a lot. That sounds pretty rough. He's been coughing up some nasty stuff, and it's been going on for a while. Now, when I hear about a round consolidation on a chest x-ray, my mind immediately jumps to a lung abscess. Those are those pesky pockets of pus in the lungs. And then there's the air-fluid level thing. That's a big clue. It's like a signpost pointing towards a lung ...

- Based on the provided symptoms and clinical findings, the correct diagnosis is B. Lung abscess. The patient's history of alcoholism suggests a heightened risk of aspiration, which could have led to the development of a lung abscess. The combination of his symptoms, including fever, chills, chest pain, and the production of malodorous sputum, along with the chest x-ray findings of a round consolida

[transformers] Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## Step 9 - Evaluate properly: the frozen 60-question audit (do NOT train on it)
Packaging first (Step 10), then in the GraphRAG repo set `LLM_MODEL = "labeltune-8b"` in `adv_graphrag.py` and re-run the audit exactly as the baseline was run. Compare base vs tuned on the same 60 questions, same grader, same session if possible - the repo's own results files note that V1 and V2 were graded in different sessions with different strictness, so part of any gap can be grader effect. Headline numbers to report: strict accuracy by T-category, and the unsupported-claim rate from the support check.

## Step 10 - Merge the adapter and package as an Ollama model (so the pipeline only changes the model name)
Mirrors the pipeline's own setup: the GraphRAG scripts run the LLM through Ollama, so the tuned model is registered as `labeltune-8b` and `adv_graphrag.py` needs only its `LLM_MODEL` changed.

In [11]:
# Merge LoRA into the base weights and save the merged model to Drive
merged = model.merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("merged model saved to:", MERGED_DIR)
del merged
torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/bnb.py:377: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

merged model saved to: /content/drive/MyDrive/GraphRAG_FineTune/merged_labeltune_8b


In [12]:
# Convert merged model -> GGUF -> Ollama model named labeltune-8b
import subprocess, os
def sh(cmd):
    print("+", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout or "")[-800:], (r.stderr or "")[-800:])
    return r.returncode

if not os.path.exists("llama.cpp"):
    sh("git clone --depth 1 https://github.com/ggerganov/llama.cpp")
sh("pip install -q -r llama.cpp/requirements.txt")
gguf_f16 = os.path.join(GGUF_DIR, "labeltune-8b-f16.gguf")
gguf_q4  = os.path.join(GGUF_DIR, "labeltune-8b-q4_k_m.gguf")
sh(f"python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {gguf_f16} --outtype f16")
sh(f"cmake -S llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release && cmake --build llama.cpp/build -j --target llama-quantize")
sh(f"./llama.cpp/build/bin/llama-quantize {gguf_f16} {gguf_q4} q4_k_m")

modelfile = f'''FROM {gguf_q4}
TEMPLATE "{{{{ if .System }}}}<|start_header_id|>system<|end_header_id|>

{{{{ .System }}}}<|eot_id|>{{{{ end }}}}{{{{ if .Prompt }}}}<|start_header_id|>user<|end_header_id|>

{{{{ .Prompt }}}}<|eot_id|>{{{{ end }}}}<|start_header_id|>assistant<|end_header_id|>

{{{{ .Response }}}}<|eot_id|>
PARAMETER temperature 0.1
PARAMETER num_ctx 8192
PARAMETER stop "<|eot_id|>"
'''
mf_path = os.path.join(GGUF_DIR, "Modelfile")
with open(mf_path, "w") as f:
    f.write(modelfile)
print("Modelfile written:", mf_path)
# Ollama must be installed and serving (same install steps as adv_graphrag.py Step 2)
sh(f"ollama create labeltune-8b -f {mf_path}")
sh("ollama list")
print("In adv_graphrag.py set LLM_MODEL = 'labeltune-8b' and re-run the audit.")

+ git clone --depth 1 https://github.com/ggerganov/llama.cpp
 Cloning into 'llama.cpp'...

+ pip install -q -r llama.cpp/requirements.txt
  4.25.9 which is incompatible.
proto-plus 1.29.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
wandb 0.28.1 requires protobuf!=5.28.0,!=5.29.0,<8,>=5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
googleapis-common-protos 1.76.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
torchvision 0.29.1+cu130 requires torch>=2.14.0, but you have torch 2.11.0+cpu which is incompatible.

+ python llama.cpp/convert_hf_to_gguf.py /content/drive/MyDr

## What is saved in Drive, and the honest limits
Under `GraphRAG_FineTune/` in the Drive you mounted:
- `data/train_t1t8.jsonl` - the exact training set (inspect it; it is the audit trail for the leakage rule)
- `adapter_labeltune_8b/` - the LoRA adapter (small; this is the real deliverable)
- `merged_labeltune_8b/` - merged 16-bit weights (needed for GGUF)
- `gguf/` - GGUF files + Modelfile for `ollama create labeltune-8b`
- `training_summary.json` - counts, hyperparameters, data source

Limits, stated plainly:
-Dataset-only design (2026-11-16): no synthetic behaviour data. If a T-category does not improve, that is the measured result to report; any behaviour pack would be a separate, later decision.
- Medical-o1 teaches reasoning from memory; the grounded format here (answer only from the given excerpt, cite, or abstain) is what keeps that from fighting T6/T8. Do not drop the system prompt at inference time.

In [14]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [15]:
import shutil, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "NousResearch/Meta-Llama-3-8B-Instruct"
ADAPTER_DIR = "/content/drive/MyDrive/GraphRAG_FineTune/adapter_labeltune_8b"
MERGED_DIR = "/content/drive/MyDrive/GraphRAG_FineTune/merged_labeltune_8b"

shutil.rmtree(MERGED_DIR, ignore_errors=True)

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model = model.merge_and_unload()
model.save_pretrained(MERGED_DIR, safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.save_pretrained(MERGED_DIR)
print("merged model saved in bf16, no quantisation")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3430: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (50GB default)
  return old_embeddings


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

merged model saved in bf16, no quantisation


In [ ]:
import os, json
d = "/content/drive/MyDrive/GraphRAG_FineTune/merged_labeltune_8b"
for f in sorted(os.listdir(d)):
    print(f, round(os.path.getsize(os.path.join(d, f)) / 1e9, 2), "GB")
cfg = json.load(open(os.path.join(d, "config.json")))
print("quantisation in config:", "quantization_config" in cfg)

### Final testing

In [16]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MERGED_DIR = "/content/drive/MyDrive/GraphRAG_FineTune/merged_labeltune_8b"
ABSTAIN_TEXT = "INSUFFICIENT FDA DATA: The retrieved label text does not contain this information."
SYSTEM_PROMPT = ("You answer ONLY from the FDA drug label excerpts below.\nRULES:\n"
    "1. Use only the excerpts. Never use outside knowledge.\n"
    "2. Write short bullet points. Every bullet ends with its source tag(s), e.g. [S2] or [S1][S3].\n"
    "3. Keep drugs separate: start each bullet with the drug name.\n"
    f"4. If the excerpts do not answer the question, reply exactly: {ABSTAIN_TEXT}\n"
    "5. Give no medical advice beyond what the label says.")

def shorten(text, words):
    w = str(text).split()
    return " ".join(w[:words]) + (" ..." if len(w) > words else "")

tok = AutoTokenizer.from_pretrained(MERGED_DIR)
model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, dtype=torch.bfloat16, device_map="auto")
model.config.use_cache = True
model.eval()
print("merged model loaded on:", next(model.parameters()).device)

rows = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en")["train"].shuffle(seed=42)

passed = 0
for r in [rows[-3], rows[-2], rows[-1]]:
    user = f"EXCERPTS:\n[S1] (medical-o1 | response) {shorten(r['Response'], 220)}\n\nQUESTION: {r['Question']}\n\nANSWER:\n"
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=300, temperature=0.1, do_sample=True,
                             pad_token_id=tok.eos_token_id)
    ans = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    ok = ans.lstrip().startswith("Reasoning:") and "[S1]" in ans
    passed += ok
    print("Q:", r["Question"][:100])
    print(ans)
    print("format check:", "PASS" if ok else "FAIL")
    print("=" * 80)

print(f"RESULT: {passed}/3 probes in the trained format ->",
      "FINE-TUNED AND READY" if passed >= 2 else "FORMAT NOT LEARNED - check the merge")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

merged model loaded on: cuda:0


[transformers] Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: A 47-year-old alcoholic man presents to the office for a 72-hour history of intense right hemithorax
Reasoning: Alright, let's think this through. We've got a 47-year-old guy who's been drinking a lot and now he's got this intense pain on the right side of his chest. That's been going on for three days. He's also got a fever, chills, and he's sweating a lot. These symptoms started about 10 days ago. He's been coughing a lot and the sputum is really foul-smelling. That's definitely not good. Now, looking at the chest x-ray, there's a round consolidation with an air-fluid level in the middle part of his right lung. That's interesting. This kind of finding makes me think of a lung abscess. It's like a pocket of pus ...

- Based on the provided symptoms and clinical findings, the correct diagnosis is B. Lung abscess. The patient's history of alcoholism suggests a heightened risk of aspiration, which could have led to the development of a lung abscess. The combination of his symptoms, in

[transformers] Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: A 25-year-old woman has been experiencing fever for one month, along with ataxia and headache. Brain
Reasoning: Alright, let's think this through. We've got a 25-year-old woman who's been dealing with a fever for a month, plus she's experiencing ataxia and headaches. That's quite a while to be in a fever, so something chronic is probably going on. Now, looking at the brain imaging, we see dilated ventricles and basal exudates. Hmm, that's interesting. Dilated ventricles often point towards something affecting the cerebrospinal fluid flow or pressure, like meningitis. And basal exudates? Those are typically seen in chronic infections. So, what could this be? Tuberculous meningitis comes to mind because it's known for causing chronic symptoms and affecting the basal cisterns. It's also notorious for causing ...

- In a case like this, where a 25-year-old woman presents with a prolonged fever, ataxia, headache, and brain imaging showing dilated ventricles with basal exudates, tuberculo